## Cell 1: Setup
**What this demonstrates**: How to load API keys from the environment and initialise the Anthropic and OpenAI clients that every Naive RAG pipeline depends on.
**Expected output**: ✓ Setup complete, with masked key suffixes confirming both keys loaded.

In [11]:
# ── Install dependencies ─────────────────────────────────────────────────────
# Run once; safe to re-run (pip is idempotent)
%pip install -q langchain langchain-openai langchain-anthropic langchain-community \
    chromadb python-dotenv tiktoken

# ── Standard library ─────────────────────────────────────────────────────────
import os
import time
import statistics
import pathlib
from typing import Any

# ── Third-party ───────────────────────────────────────────────────────────────
from dotenv import load_dotenv, find_dotenv          # reads .env file into os.environ
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken

# ── Load API keys ─────────────────────────────────────────────────────────────
# load_dotenv() must be called before any client is instantiated
load_dotenv(find_dotenv())

_anthropic_key = os.environ.get("ANTHROPIC_API_KEY", "")
_openai_key    = os.environ.get("OPENAI_API_KEY", "")
assert _anthropic_key, "ANTHROPIC_API_KEY not set — add it to .env"
assert _openai_key,    "OPENAI_API_KEY not set — add it to .env"

# ── Constants ─────────────────────────────────────────────────────────────────
LLM_MODEL   = "gpt-4o-mini"             # OpenAI model for generation
EMBED_MODEL = "text-embedding-3-small"  # 1536-dim OpenAI embeddings
CHROMA_DIR  = "./chroma_naive_rag"      # local vector store persistence path

# ── Client initialisation ─────────────────────────────────────────────────────
# OpenAI() reads OPENAI_API_KEY from env — key never appears in code
client: OpenAI = OpenAI()
# OpenAIEmbeddings reads OPENAI_API_KEY from env automatically
embeddings: OpenAIEmbeddings = OpenAIEmbeddings(model=EMBED_MODEL)

print("✓ Setup complete")
print(f"  LLM model:       {LLM_MODEL}")
print(f"  Embedding model: {EMBED_MODEL}")
print(f"  Anthropic key:   ...{_anthropic_key[-4:]}")
print(f"  OpenAI key:      ...{_openai_key[-4:]}")



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✓ Setup complete
  LLM model:       claude-sonnet-4-6
  Embedding model: text-embedding-3-small
  Anthropic key:   ...-...
  OpenAI key:      ...-...


## Cell 2: Data
**What this demonstrates**: How to load the synthetic fintech policy document that serves as the knowledge base for this pipeline.
**Expected output**: Document name, character/word count, and the first 200 characters of the loan policy text.

In [3]:
# ── Locate the sample document ────────────────────────────────────────────────
# Try two paths: when the notebook runs from its own directory, or from repo root.
_candidates = [
    pathlib.Path("../../shared/sample_data/fintech_policy.txt"),
    pathlib.Path("shared/sample_data/fintech_policy.txt"),
]
SAMPLE_DATA = next((p for p in _candidates if p.exists()), None)
assert SAMPLE_DATA is not None, (
    "Cannot find fintech_policy.txt. "
    "Run the notebook from the repo root or from modules/01_naive_rag/."
)

document_text: str = SAMPLE_DATA.read_text(encoding="utf-8")

print(f"Document:  {SAMPLE_DATA.name}")
print(f"Size:      {len(document_text):,} characters  |  ~{len(document_text.split()):,} words")
print()
print("First 200 characters:")
print("-" * 52)
print(document_text[:200])
print("-" * 52)
print()
print("Why this document?")
print("  fintech_policy.txt is a structured loan policy with numbered sections,")
print("  eligibility tables, and cross-references between sections — exactly the")
print("  kind of internal document compliance teams query every day. It tests")
print("  whether chunking can preserve section context across split boundaries.")


Document:  fintech_policy.txt
Size:      12,853 characters  |  ~1,701 words

First 200 characters:
----------------------------------------------------
MERIDIAN BANK — CONSUMER LENDING POLICY
Version 4.2 | Effective Date: January 1, 2025
Classification: Internal Use Only

----------------------------------------------------

Why this document?
  fintech_policy.txt is a structured loan policy with numbered sections,
  eligibility tables, and cross-references between sections — exactly the
  kind of internal document compliance teams query every day. It tests
  whether chunking can preserve section context across split boundaries.


## Cell 3: Core — Chunking and Indexing
**What this demonstrates**: The two indexing steps of Naive RAG: splitting the document into retrievable chunks, embedding each chunk, and persisting the index in a local Chroma vector store.
**Expected output**: Number of chunks created, index build time, and previews of the first three chunks showing where the splitter made cuts.

In [9]:
# ── Step 1: Chunk the document ────────────────────────────────────────────────
def chunk_document(text: str, chunk_size: int = 500, chunk_overlap: int = 50) -> list[str]:
    """Split a document into overlapping text chunks for embedding and retrieval.

    Args:
        text: Raw document string.
        chunk_size: Target chunk length in characters (not tokens).
        chunk_overlap: Characters of overlap between adjacent chunks — prevents an
            answer from being cut cleanly in half at a boundary.

    Returns:
        List of chunk strings ordered by appearance in the source document.
    """
    splitter = RecursiveCharacterTextSplitter(
        # Try each separator in priority order, falling back to the next if the
        # resulting chunk would still exceed chunk_size.
        separators=["\n\n", "\n", ". ", " ", ""],
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,  # character count — swap for token-based splitting
    )
    return splitter.split_text(text)


# ── Step 2: Embed chunks and build the vector index ───────────────────────────
def build_index(
    chunks: list[str],
    collection_name: str = "naive_rag_policy",
) -> Chroma:
    """Embed each chunk and persist the index in a local Chroma vector store.

    Args:
        chunks: Text chunks produced by chunk_document().
        collection_name: Chroma collection identifier. Change to force a fresh index.

    Returns:
        Populated Chroma instance ready for similarity_search_with_score().
    """
    vectorstore = Chroma.from_texts(
        texts=chunks,
        embedding=embeddings,          # MUST use the same model at query time
        collection_name=collection_name,
        persist_directory=CHROMA_DIR,
        # cosine distance → similarity = 1 − distance gives a clean [0, 1] score
        collection_metadata={"hnsw:space": "cosine"},
    )
    return vectorstore


# ── Execute indexing and report ───────────────────────────────────────────────
t0 = time.perf_counter()
chunks: list[str] = chunk_document(document_text)
vectorstore: Chroma = build_index(chunks)
index_ms = (time.perf_counter() - t0) * 1000

print(f"Chunks created:  {len(chunks)}")
print(f"Chunk size:      500 chars  |  overlap: 50 chars")
print(f"Index built in:  {index_ms:.0f} ms")
print()
print("First 3 chunks (truncated to 120 chars):")
for i, chunk in enumerate(chunks[:3], 1):
    preview = chunk[:120].replace("\n", " ")
    print(f"  [{i:02d}] {preview!r}")


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-.... You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

## Cell 4: Run — Query and Generate
**What this demonstrates**: The full Naive RAG query path: embed the question, retrieve the top-5 most similar chunks by cosine similarity, and generate a grounded answer with Claude.
**Expected output**: The five retrieved chunks with their similarity scores, followed by the LLM-generated answer and a latency breakdown.

In [ ]:
# ── System prompt ────────────────────────────────────────────────────────────
# Explicit instruction to stay grounded in context and signal when it can't answer
SYSTEM_PROMPT = (
    "You are a financial analyst assistant helping staff with policy questions. "
    "Answer using ONLY the provided context excerpts. "
    "If the context does not contain enough information to answer, respond with: "
    "'I cannot find this information in the provided documents.' "
    "Always cite the relevant section when answering."
)


# ── Step 3: Retrieve top-k chunks ────────────────────────────────────────────
def retrieve(query: str, k: int = 5) -> list[tuple[Any, float]]:
    """Retrieve the top-k chunks most semantically similar to the query.

    Args:
        query: Natural language question.
        k: Number of results to return.

    Returns:
        List of (Document, similarity_score) tuples, sorted by descending score.
        Scores are in [0, 1] — 1.0 is a perfect cosine match.
    """
    # similarity_search_with_score returns (Document, cosine_distance) pairs.
    # With hnsw:space=cosine: distance = 1 − cosine_similarity
    # so: similarity_score = 1 − distance
    raw = vectorstore.similarity_search_with_score(query, k=k)
    return [(doc, 1.0 - dist) for doc, dist in raw]


# ── Step 4: Generate a grounded answer ───────────────────────────────────────
def generate(query: str, context_chunks: list[tuple[Any, float]]) -> str:
    """Generate an answer grounded in the retrieved context chunks.

    Args:
        query: The user's question.
        context_chunks: Retrieved (Document, score) pairs from retrieve().

    Returns:
        LLM-generated answer string.
    """
    # Label each chunk with rank and score so the model can reference them
    context = "\n\n".join(
        f"[Chunk {i+1} | relevance={score:.3f}]\n{doc.page_content}"
        for i, (doc, score) in enumerate(context_chunks)
    )
    response = client.chat.completions.create(
        model=LLM_MODEL,
        max_tokens=512,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
        ],
    )
    return response.choices[0].message.content


# ── Execute the full pipeline ─────────────────────────────────────────────────
query = "What are the eligibility requirements for a personal loan?"

t_ret = time.perf_counter()
retrieved: list[tuple[Any, float]] = retrieve(query)
retrieval_ms = (time.perf_counter() - t_ret) * 1000

t_gen = time.perf_counter()
answer: str = generate(query, retrieved)
generation_ms = (time.perf_counter() - t_gen) * 1000

# ── Print results ─────────────────────────────────────────────────────────────
print(f"Query: {query!r}")
print()
print(f"Retrieved {len(retrieved)} chunks:")
for i, (doc, score) in enumerate(retrieved, 1):
    preview = doc.page_content[:100].replace("\n", " ")
    print(f"  [{i}] score={score:.3f}  {preview!r}...")

print()
print("=" * 65)
print("ANSWER")
print("=" * 65)
print(answer)
print("=" * 65)
print()
print(f"Retrieval: {retrieval_ms:.0f} ms  |  Generation: {generation_ms:.0f} ms  |  Total: {retrieval_ms + generation_ms:.0f} ms")


## Cell 5: Inspect — Observability
**What this demonstrates**: How to examine chunk boundaries, score distributions, and the token budget to understand where Naive RAG is likely to fail before it does so in production.
**Expected output**: Boundary previews showing actual split points, a score histogram with a quality flag, and a token cost breakdown per pipeline component.

In [ ]:
# ── Chunk boundary inspection ─────────────────────────────────────────────────
def show_chunk_boundaries(all_chunks: list[str], n: int = 4, ctx: int = 50) -> None:
    """Print the tail of each chunk and the head of the next to reveal split quality.

    Args:
        all_chunks: Full list of document chunks produced by chunk_document().
        n: Number of boundaries to display.
        ctx: Characters of tail/head to show on each side of the boundary.
    """
    print(f"Chunk boundary inspection (showing {n} split points):")
    print("-" * 65)
    for i in range(min(n, len(all_chunks) - 1)):
        # Replace newlines with ↵ so the boundary is legible on one line
        tail = all_chunks[i][-ctx:].replace("\n", "↵")
        head = all_chunks[i + 1][:ctx].replace("\n", "↵")
        print(f"  [{i}→{i+1}]  ...{tail!r}")
        print(f"           {head!r}...")
        print()


# ── Score distribution ────────────────────────────────────────────────────────
def show_score_distribution(results: list[tuple[Any, float]]) -> None:
    """Print a text histogram of retrieval scores and flag low-spread results.

    A spread < 0.05 means all retrieved chunks score nearly identically,
    so swapping the top-1 chunk for any other would barely change quality —
    a warning sign that retrieval is not discriminating well.

    Args:
        results: Retrieved (Document, score) pairs.
    """
    scores = [s for _, s in results]
    spread = max(scores) - min(scores)
    print(f"Score distribution  (max={max(scores):.3f}  min={min(scores):.3f}  spread={spread:.3f}):")
    for i, (_, score) in enumerate(scores, 1):
        bar = "█" * int(score * 40)  # bar width proportional to score
        print(f"  [{i}] {score:.3f} {bar}")
    if spread < 0.05:
        print("  ⚠  Spread < 0.05 — top-k scores are nearly equal; retrieval may be unreliable")
    print()


# ── Token counting ────────────────────────────────────────────────────────────
def count_tokens(text: str, encoding: str = "cl100k_base") -> int:
    """Count tokens using the cl100k_base tiktoken encoding (GPT-4 / Ada-v2).

    Args:
        text: Input string.
        encoding: tiktoken encoding name.

    Returns:
        Integer token count.
    """
    enc = tiktoken.get_encoding(encoding)
    return len(enc.encode(text))


# ── Run inspection ────────────────────────────────────────────────────────────
show_chunk_boundaries(chunks)
show_score_distribution(retrieved)

# Assemble the context block as it was sent to the LLM
context_text = "\n\n".join(doc.page_content for doc, _ in retrieved)

sys_tok = count_tokens(SYSTEM_PROMPT)
ctx_tok = count_tokens(context_text)
q_tok   = count_tokens(query)
ans_tok = count_tokens(answer)
total   = sys_tok + ctx_tok + q_tok

print("Token budget:")
print(f"  System prompt:        {sys_tok:>5} tokens")
print(f"  Context (5 chunks):   {ctx_tok:>5} tokens")
print(f"  Query:                {q_tok:>5} tokens")
print(f"  ─────────────────────────────────")
print(f"  Total input:          {total:>5} tokens")
print(f"  Generated answer:     {ans_tok:>5} tokens")
print()
print(f"  Approx cost @ $3/M input + $15/M output:")
input_cost  = total  / 1_000_000 * 3.0
output_cost = ans_tok / 1_000_000 * 15.0
print(f"  Input:  ${input_cost:.5f}  |  Output: ${output_cost:.5f}  |  Total: ${input_cost + output_cost:.5f}")


## Cell 6: Fintech Application — Compliance Q&A
**What this demonstrates**: Naive RAG applied to a compliance question about loan default procedures, with source citation tracking and an explicit gap analysis showing what the pipeline cannot do.
**Expected output**: Retrieved section excerpts with scores, a compliance answer citing specific sections, and a printed list of the gaps that later modules fix.

In [ ]:
# ── Compliance-specific prompt ────────────────────────────────────────────────
# Stricter grounding requirement: must cite section number and quote policy language
COMPLIANCE_PROMPT = (
    "You are a compliance officer assistant. "
    "Answer the compliance question using only the provided policy excerpts. "
    "Cite the specific section number and quote the relevant policy language verbatim. "
    "If the answer is not present in the excerpts, state that clearly."
)


def run_compliance_query(
    compliance_query: str,
    k: int = 5,
) -> dict[str, Any]:
    """Execute the full Naive RAG pipeline for a compliance question.

    Args:
        compliance_query: The compliance or policy question to answer.
        k: Number of context chunks to retrieve.

    Returns:
        Dict with keys: query, answer, chunks (list of (doc, score) tuples),
        retrieval_ms, generation_ms.

    Fintech note:
        In a production compliance system, you would additionally verify that
        the cited section numbers match the retrieved content (hallucination check)
        and log every query/answer pair for audit trail purposes.
    """
    t0 = time.perf_counter()
    raw = vectorstore.similarity_search_with_score(compliance_query, k=k)
    ret_ms = (time.perf_counter() - t0) * 1000

    # Convert cosine distance → similarity score
    results: list[tuple[Any, float]] = [(doc, 1.0 - dist) for doc, dist in raw]

    context = "\n\n".join(
        f"[Section excerpt {i+1} | relevance={score:.3f}]\n{doc.page_content}"
        for i, (doc, score) in enumerate(results)
    )
    t1 = time.perf_counter()
    response = client.chat.completions.create(
        model=LLM_MODEL,
        max_tokens=600,
        messages=[
            {"role": "system", "content": COMPLIANCE_PROMPT},
            {
                "role": "user",
                "content": f"Policy excerpts:
{context}

Compliance question: {compliance_query}",
            }
        ],
    )
    gen_ms = (time.perf_counter() - t1) * 1000

    return {
        "query": compliance_query,
        "answer": response.content[0].text,
        "chunks": results,
        "retrieval_ms": ret_ms,
        "generation_ms": gen_ms,
    }


# ── Run compliance Q&A ────────────────────────────────────────────────────────
result = run_compliance_query(
    "What happens if a borrower defaults on their personal loan?"
)

print("=" * 65)
print("COMPLIANCE Q&A  —  Default & Remediation")
print("=" * 65)
print(f"\nQuestion: {result['query']}")
print(f"\nSource excerpts ({len(result['chunks'])} chunks retrieved):")
for i, (doc, score) in enumerate(result["chunks"], 1):
    # Use the first non-blank line of the chunk as a section label
    label = next((l.strip() for l in doc.page_content.split("\n") if l.strip()), "")
    print(f"  [{i}] score={score:.3f}  '{label[:62]}'")

print(f"\nCompliance answer:")
print("-" * 65)
print(result["answer"])
print("-" * 65)
print(f"\nLatency: retrieval={result['retrieval_ms']:.0f} ms  |  generation={result['generation_ms']:.0f} ms")

# ── Gap analysis — motivates the next modules ─────────────────────────────────
print()
print("=" * 65)
print("WHAT NAIVE RAG DOES NOT DO  (gaps fixed by later modules)")
print("=" * 65)
print("  ✗ No query rewriting   — exact phrasing determines recall [→ Module 02]")
print("  ✗ No reranking         — cosine order is final            [→ Module 02]")
print("  ✗ No relevance grading — low-score chunks enter context   [→ Module 16]")
print("  ✗ No cross-section synthesis — misses multi-section answers [→ Module 10]")
print("  ✗ No 'I don't know'    — always generates, even blind     [→ Module 17]")
